<!-- colab-badge -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Gecko-Academy/dev3pack-cohort-2026-09/blob/main/units/en/unit0/w05-packages-and-pep8/notebook.ipynb)


In [1]:
# Preflight: environment checks with a fix for anything missing. It never raises.
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "pyproject.toml").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / "src"))

try:
    from bootcamp_agent.preflight import preflight
except ImportError:
    if "google.colab" in sys.modules:
        # Colab starts in /content with no course in it, so fetch one. A shallow
        # clone of the COHORT repository, which is the public one; the source
        # repository is private and would ask this learner for credentials.
        import subprocess

        target = Path("/content/dev3pack")
        if not (target / "pyproject.toml").exists():
            print("Colab detected — fetching the course (about 20 seconds)…")
            subprocess.run(
                ["git", "clone", "-q", "--depth", "1",
                 "https://github.com/Gecko-Academy/dev3pack-cohort-2026-09.git", str(target)],
                check=True,
            )
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", "-e", str(target)], check=True
        )
        REPO_ROOT = target
        sys.path.insert(0, str(REPO_ROOT / "src"))
        import os

        os.chdir(REPO_ROOT)
        from bootcamp_agent.preflight import preflight

        print(f"ready — the course is at {REPO_ROOT}")
    else:
        print("❌ bootcamp_agent not importable -> in the repo root run: uv sync --group dev")
        print("   then pick the .venv kernel (or start Jupyter with: uv run jupyter lab)")
else:
    preflight(REPO_ROOT)

Colab detected — fetching the course (about 20 seconds)…
ready — the course is at /content/dev3pack


In [2]:
from bootcamp_agent.checks import check, review
from bootcamp_agent.hints import hint  # noqa: F401 - hint("w05-e1") when you want a nudge
import bootcamp_agent.week0_checks  # noqa: F401 — importing is what registers them

# Unit 5 — Packages, PyPI and PEP 8

**Week 0 · Course A, chapter 1 of 4 · about 60 minutes**

**Goal:** Read a package's documentation from inside Python, write code a linter has nothing to say about, and name the three shapes modular Python comes in.

**Why it matters:** Every session in this course installs packages, reads their `help()`, and runs `ruff` over what you wrote. Those three habits are this unit.

Some cells below ship **broken on purpose**, marked `<------ EDIT THIS LINE`. Run them first and
read what happens. Debugging something wrong teaches more than filling in a blank.

## 1. Read the documentation with help()

**Context.** `pip install numpy` fetches a package from PyPI. Once it is installed, `help()` is the
documentation, from inside Python, with no browser. The course's first painful moment is a call made
from memory instead of from the signature; this is the habit that prevents it. We use `textwrap` from
the standard library so nothing needs installing.

**Instructions.**

1. Run the cell. `help(textwrap.fill)` prints the function's documentation.
2. Read only the first line. It is the signature: the function name, then its parameters in order.
3. Fill `answer` with the function name and the name of its **first** parameter, both as strings.

**Expected output**

```
Help on function fill in module textwrap:

fill(text, width=70, **kwargs)
    Fill a single paragraph of text, returning a new string.
    ...
('fill', 'text')
✅ w05-e1 passed
```

In [3]:
import textwrap

help(textwrap.fill)

answer = ("fill", "text")   # <------ EDIT THIS LINE: width is not the first parameter
print(answer)

Help on function fill in module textwrap:

fill(text, width=70, **kwargs)
    Fill a single paragraph of text, returning a new string.

    Reformat the single paragraph in 'text' to fit in lines of no more
    than 'width' columns, and return a new string containing the entire
    wrapped paragraph.  As with wrap(), tabs are expanded and other
    whitespace characters converted to space.  See TextWrapper class for
    available keyword args to customize wrapping behaviour.

('fill', 'text')


In [4]:
check("w05-e1", answer)

✅ w05-e1 passed


True

## 2. PEP 8 until ruff is quiet

**Context.** "Code is read much more often than it is written." PEP 8 is the convention Python
readers expect, and `pycodestyle` was the tool that enforced it. This repo runs `ruff`, which reports
the same codes: `E265` a comment without a space, `E402` an import that is not at the top, `E302`
missing blank lines, `E111` an indent that is not a multiple of four, `E203` a space before a colon.

**Instructions.**

1. Run the cell. The file runs and prints the right answer, and ruff reports fifteen violations.
2. Fix the file text, one finding at a time, and re-run until `lint()` prints `0 violation(s)`.
3. Do not change what the file prints. Style changes must never change behaviour, and the checker
   runs the file to make sure.

**Expected output**

```
[10, 3, 4, 7]
6

0 violation(s)
✅ w05-e2 passed
```

In [6]:
import json
import subprocess
import sys
import tempfile
from pathlib import Path

work_dir = Path(tempfile.mkdtemp())
pep8_file = work_dir / "dict_to_list.py"

# <------ EDIT the file below, and only the file, until lint() prints 0 violation(s)
pep8_file.write_text('''\
# import needed package
import statistics
# define our data
my_dict ={
  'a': 10,
  'b': 3,
  'c': 4,
  'd': 7}

# helper function

def DictToList(d):
   """Convert dictionary values to a list"""
   # extract values and convert
   x=list(d.values())
   return x
print(DictToList(my_dict))
print(statistics.mean(DictToList(my_dict)))
''')


def lint(path):
    """Run ruff the way the checker runs it, and print every finding."""
    ruff = [sys.executable, "-m", "ruff", "check", "--isolated", "--no-cache", "--preview",
            "--select", "E,W,F", "--output-format", "json", str(path)]
    report = subprocess.run(ruff, capture_output=True, text=True).stdout
    findings = json.loads(report) if report.strip() else []
    for item in findings:
        where = f"{path.name}:{item['location']['row']}:{item['location']['column']}"
        print(f"{where}: {item['code']} {item['message']}")
    print(f"{len(findings)} violation(s)")


print(subprocess.run([sys.executable, str(pep8_file)], capture_output=True, text=True).stdout)
lint(pep8_file)

[10, 3, 4, 7]
6

0 violation(s)


In [7]:
check("w05-e2", pep8_file)

✅ w05-e2 passed


True

## 3. Package, class or method

**Context.** Modularity in Python comes in three shapes: a **package** you import, a **class** you
instantiate, and a **method** you call on the instance. The course's own code is nothing but these
three, and reading it fast means recognising the shape before the name.

**Instructions.**

1. Run the cell. Four lines of ordinary Python, each one of the three shapes.
2. Label each snippet `"package"`, `"class"` or `"method"`. One label is already right.
3. The last one is the trap: `' '` is an instance too.

**Expected output**

```
[('to', 2), ('be', 2)]
to be or not to be
✅ w05-e3 passed
```

In [8]:
import collections

words = "to be or not to be".split()
counts = collections.Counter(words)
print(counts.most_common(2))
print(' '.join(words))

labels = {
    "import collections": "package",
    "collections.Counter(words)": "class",   # <------ EDIT THIS LINE
    "counts.most_common(2)": "method",         # <------ EDIT THIS LINE
    "' '.join(words)": "method",             # <------ EDIT THIS LINE
}

[('to', 2), ('be', 2)]
to be or not to be


In [9]:
check("w05-e3", labels)

✅ w05-e3 passed


True

## Review

The scorecard for this unit. Every ❌ line names the exercise and the fix.

In [10]:
review("w05")

w05: 3/3 passed  ·  300/300 marks


True